In [2]:
#import necessary libraries

import pandas as pd
import numpy as np
import re

from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sentence_transformers import CrossEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import ndcg_score

About Dataset:


1. documents.csv

Contains the agricultural knowledge base that the retrieval model searches.

Columns

* document_id – Unique identifier for each document.
* title – Title of the agricultural document.
* text – Main content of the document.
* source – Organization that published the document.
* crop – Crop associated with the document.
* country – Country or region the document applies to.
* origin – Indicates whether the document is original or synthetic.
* source_url – Link to the original document.
* license – Usage license for the document.


2. train_queries.csv

Contains the training questions that simulate the types of questions a farmer may ask.

Columns

* query_id uniquely identifies each training query.
* query contains the farmer's question.
* positive_docs contains the IDs of documents that are relevant to that query.


3. qrels_train.csv

Contains the relevance judgments (also called qrels) used to evaluate how well a retrieval model ranks documents for each training query.

Columns
* query_id – Identifies the training query.
* document_id – Identifies a document associated with that query.
* relevance – Indicates how relevant the document is to the query.


4. test_queries.csv

Contains the unseen farmer questions used to evaluate the retrieval system.

Columns
* query_id – Uniquely identifies each test query.
* query – Contains the farmer's question for which the model must retrieve the most relevant documents.

In [ ]:
# Load the all datasets

documents = pd.read_csv('documents.csv')

train_queries = pd.read_csv('train_queries.csv')

qrels_train = pd.read_csv('qrels_train.csv')

test_queries = pd.read_csv('test_queries.csv')


* Descriptive analysis

In [4]:
documents.head()

,document_id,title,text,source,crop,country,origin,source_url,license
0,1,Drought and erratic rainfall: the risk to crops,Drought and erratic rainfall and its impact on...,FAO,(general),Kenya,synthetic,NaN,synthetic (CC0)
1,2,Adapting to drought and erratic rainfall (Guin...,Adapting to drought and erratic rainfall in th...,IITA,(general),Tanzania,synthetic,NaN,synthetic (CC0)
2,3,Adapting to drought and erratic rainfall (High...,Adapting to drought and erratic rainfall in th...,ICRISAT,(general),Nigeria,synthetic,NaN,synthetic (CC0)
3,4,Adapting to drought and erratic rainfall (Humi...,Adapting to drought and erratic rainfall in th...,FAO,(general),Mali,synthetic,NaN,synthetic (CC0)
4,5,Adapting to drought and erratic rainfall (Sahel),Adapting to drought and erratic rainfall in th...,FAO,(general),Tanzania,synthetic,NaN,synthetic (CC0)


In [5]:
documents.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 695 entries, 0 to 694
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   document_id  695 non-null    int64 
 1   title        695 non-null    object
 2   text         695 non-null    object
 3   source       695 non-null    object
 4   crop         695 non-null    object
 5   country      695 non-null    object
 6   origin       695 non-null    object
 7   source_url   58 non-null     object
 8   license      695 non-null    object
dtypes: int64(1), object(8)
memory usage: 49.0+ KB


In [6]:
# check for duplicates
documents['text'].duplicated().sum()

np.int64(0)

* 'source_url' column in 'documents' dataset has many missing values but is not used for retrieval, so we'll leave it as is
* No duplicated texts
* Text appears clean and well formatted.

In [7]:
train_queries.head()

,query_id,query,positive_docs
0,1,How do I cope with flooding and excess rain on...,8 9 10 7 6
1,2,How can I adapt my farming to flooding and exc...,8 9 10 7 6
2,3,How does flooding and excess rain affect my cr...,6 8 9 10 7
3,4,What is the risk of flooding and excess rain t...,6 8 9 10 7
4,5,How do I manage bacterial leaf blight in rice?,40 36 38


In [8]:
train_queries.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 308 entries, 0 to 307
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   query_id       308 non-null    int64 
 1   query          308 non-null    object
 2   positive_docs  308 non-null    object
dtypes: int64(1), object(2)
memory usage: 7.3+ KB


In [9]:
# Check for duplicated query
train_queries['query'].duplicated().sum()

np.int64(0)

* No duplicated query
* Data is clean and well structured

In [10]:
qrels_train.head()

,query_id,document_id,relevance
0,1,8,3.0
1,1,9,3.0
2,1,10,3.0
3,1,7,3.0
4,1,6,2.0


In [ ]:
qrels_train['relevance'].max()

np.float64(3.0)

In [12]:
qrels_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4194 entries, 0 to 4193
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   query_id     4194 non-null   int64  
 1   document_id  4194 non-null   int64  
 2   relevance    4194 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 98.4 KB


In [13]:
qrels_train.duplicated().sum()

np.int64(0)

* 4,194 query-document relevance pairs.
* No missing values.
* Each query can have multiple associated documents.
* Documents are assigned graded relevance scores ranging from 0 to 3, where higher values indicate greater relevance. 

In [14]:
test_queries.head()

,query_id,query
0,1001,How do I cope with drought and erratic rainfal...
1,1002,How can I adapt my farming to drought and erra...
2,1003,How does drought and erratic rainfall affect m...
3,1004,What is the risk of drought and erratic rainfa...
4,1005,How do I cope with heat stress on my farm?


In [15]:
test_queries.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   query_id  200 non-null    int64 
 1   query     200 non-null    object
dtypes: int64(1), object(1)
memory usage: 3.3+ KB


In [16]:
test_queries['query'].duplicated().sum()

np.int64(0)

* 200 test queries
* No missing values
* No duplicate queries

* Competition baseline: This baseline reportedly achieves approximately 0.55 nDCG@5 on the competition evaluation.

In [17]:

# Build a TF-IDF index over title +qrels_train.info() text
vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2)
doc_mat = vec.fit_transform(documents["title"] + ". " + documents["text"])

# Retrieve top-5 for each query -> LONG format (QueryId, DocumentId); row order = rank
q_mat = vec.transform(test_queries["query"])
sims = cosine_similarity(q_mat, doc_mat)

rows = []
for i, qid in enumerate(test_queries["query_id"]):
    top5 = sims[i].argsort()[::-1][:5]
    for rank in top5:
        rows.append({"QueryId": qid,
                     "DocumentId": int(documents.iloc[rank]["document_id"])})

In [18]:
rows

[{'QueryId': 1001, 'DocumentId': 1},
 {'QueryId': 1001, 'DocumentId': 2},
 {'QueryId': 1001, 'DocumentId': 4},
 {'QueryId': 1001, 'DocumentId': 3},
 {'QueryId': 1001, 'DocumentId': 5},
 {'QueryId': 1002, 'DocumentId': 1},
 {'QueryId': 1002, 'DocumentId': 2},
 {'QueryId': 1002, 'DocumentId': 4},
 {'QueryId': 1002, 'DocumentId': 3},
 {'QueryId': 1002, 'DocumentId': 5},
 {'QueryId': 1003, 'DocumentId': 1},
 {'QueryId': 1003, 'DocumentId': 2},
 {'QueryId': 1003, 'DocumentId': 4},
 {'QueryId': 1003, 'DocumentId': 3},
 {'QueryId': 1003, 'DocumentId': 5},
 {'QueryId': 1004, 'DocumentId': 1},
 {'QueryId': 1004, 'DocumentId': 2},
 {'QueryId': 1004, 'DocumentId': 4},
 {'QueryId': 1004, 'DocumentId': 3},
 {'QueryId': 1004, 'DocumentId': 5},
 {'QueryId': 1005, 'DocumentId': 11},
 {'QueryId': 1005, 'DocumentId': 12},
 {'QueryId': 1005, 'DocumentId': 13},
 {'QueryId': 1005, 'DocumentId': 14},
 {'QueryId': 1005, 'DocumentId': 15},
 {'QueryId': 1006, 'DocumentId': 11},
 {'QueryId': 1006, 'DocumentId':

BM25

In [19]:
# make copy of documents
doc1 = documents.copy()
train_queries1 = train_queries.copy()

In [20]:
def normalizer (text):
    text = text.lower()
    # remove html
    text = re.sub(r'http\S+', '', text)
    #remove punctuations
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    # remove whitespaces
    text = re.sub(r'\s+', ' ', text)


    return text.strip()

In [21]:
# normalze all douments
doc1["title"] = documents["title"].apply(normalizer)

doc1["text"] = documents["text"].apply(normalizer)

train_queries1["query"] = train_queries["query"].apply(normalizer)


# tokenize all documents
doc_tokens = doc1['text'].apply(str.split).tolist()

bm25 = BM25Okapi(doc_tokens)

In [22]:
all_bm25_scores = []

# loop through each training queries
for ind, row in train_queries1.iterrows():
    query_id = row['query_id']
    query = row['query']

    # tokenize each query
    query_tokens = query.split()

    # compute a bm25score for eeach query_token
    bm25_score = bm25.get_scores(query_tokens)
    all_bm25_scores.append(bm25_score)

# convert to numpy array
all_bm25_scores = np.array(all_bm25_scores)
   

In [23]:
all_bm25_scores.shape

(308, 695)

In [25]:
# extarct the first query and its top 5 documnets

query1 = train_queries1['query'].iloc[0]
print(f"Query: {query1}")

#query1 top5 bm25 scores
top5_indices = np.argsort(all_bm25_scores[0])[::-1] [:5]

# extract documents
for i in top5_indices:
    print(
        f"Doc_id: {doc1['document_id'].iloc[i]}, "
        f" Title: {doc1['title'].iloc[i]}"
    )    

Query: how do i cope with flooding and excess rain on my farm
Doc_id: 6,  Title: flooding and excess rain the risk to crops
Doc_id: 34,  Title: how bacterial leaf blight spreads in rice
Doc_id: 9,  Title: adapting to flooding and excess rain humid forest
Doc_id: 7,  Title: adapting to flooding and excess rain guinea savanna
Doc_id: 10,  Title: adapting to flooding and excess rain sahel


In [26]:
#nCDG5 function
def ncdg5(scores, queries, qrels, docs, k=5):
    """
    Calculate mean nDCG@5 for a retrieval system.

    scores:
        Retrieval scores with shape (number_of_queries, number_of_documents)

    queries:
        DataFrame containing query_id.

    qrels:
        DataFrame containing query_id, document_id, and relevance.

    documents:
        DataFrame containing document_id.
    """

    # create ground-truth relevance matrix
    y_true = np.zeros(scores.shape)

    # map query ids to rows
    query_to_row = {
        query_id: i
        for i, query_id in enumerate(queries["query_id"])
    }

    # map document ids to columns
    doc_to_col = {
        doc_id: i
        for i, doc_id in enumerate(docs["document_id"])
    }

    # fill relevance matrix
    for _, row in qrels.iterrows():
        query_id = row["query_id"]
        document_id = row["document_id"]
        relevance = row["relevance"]

        if query_id in query_to_row and document_id in doc_to_col:
            y_true[
                query_to_row[query_id],
                doc_to_col[document_id]
            ] = relevance

   
    return ndcg_score(y_true, scores, k=k)

In [27]:
print(f'BM25 nDCG@5: {ncdg5(all_bm25_scores, train_queries, qrels_train, documents):.3f}')

BM25 nDCG@5: 0.419


Dense Retrieval(sentence-transformer embeddings + cosine similarity)

> Sentence transformer used: all-MiniLM-L6-v2

In [28]:
# make copy of documents
doc2 = documents.copy()
train_queries2 = train_queries.copy()

In [29]:
# load model
model1 = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 137.72it/s]


In [30]:
# encode documents
doc_emb = model1.encode(list(doc2["title"] + ". " + doc2["text"]))

# encode training set of the training_query
query_emb = model1.encode(list(train_queries2['query']))

# compute cosine similarity
cos_sim_score = cosine_similarity(query_emb, doc_emb)

In [31]:
print(doc_emb.shape)
print(query_emb.shape)
print(cos_sim_score.shape)

(695, 384)
(308, 384)
(308, 695)


In [32]:
dense1_ndcg5 = ncdg5(
    cos_sim_score,
    train_queries2,
    qrels_train,
    doc2
)

print(f"Dense1 Retrieval nDCG@5: {dense1_ndcg5:.3f}")

Dense1 Retrieval nDCG@5: 0.708


* Initial dense retrieval (Dense1 Retrieval) used all-MiniLM-L6-v2, a Sentence Transformer model, to generate dense embeddings and achieved an nDCG@5 of 0.708. This served as the baseline for subsequent dense-retrieval.

> Sentence transformer used: e5-base-v2

In [ ]:
model2 = SentenceTransformer('intfloat/e5-base-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1075.40it/s]


In [ ]:
# encode documents
doc_emb = model2.encode(list(doc2["title"] + ". " + doc2["title"] + ". " + doc2["text"]))

# encode training set of the training_query
query_emb = model2.encode(list(train_queries2['query']))

# compute cosine similarity
cos_sim_score = cosine_similarity(query_emb, doc_emb)

In [ ]:
print(doc_emb.shape)
print(query_emb.shape)
print(cos_sim_score.shape)

(695, 768)
(308, 768)
(308, 695)


In [ ]:
dense_ndcg5 = ncdg5(
    cos_sim_score,
    train_queries2,
    qrels_train,
    doc2
)

print(f"Dense Retrieval nDCG@5: {dense_ndcg5:.3f}")

Dense Retrieval nDCG@5: 0.791


In [ ]:
# extarct the first query and its top 5 documnets
q1 = np.argsort(cos_sim_score[0])[::-1][:5]
q1


print(f"Query: {train_queries2['query'].iloc[0]}")
for ind in q1:
    print(f"Doc_id: {doc2['document_id'][ind]}, Title: {doc2['title'][ind]}")

Query: How do I cope with flooding and excess rain on my farm?
Doc_id: 8, Title: Adapting to flooding and excess rain (Highlands)
Doc_id: 7, Title: Adapting to flooding and excess rain (Guinea savanna)
Doc_id: 9, Title: Adapting to flooding and excess rain (Humid forest)
Doc_id: 6, Title: Flooding and excess rain: the risk to crops
Doc_id: 10, Title: Adapting to flooding and excess rain (Sahel)


After establishing the initial dense-retrieval baseline, `intfloat/e5-base-v2` was evaluated to investigate whether a stronger embedding model and different document representations could improve retrieval performance.

Four document representations were compared using the **250-query development/train set**:

| Document representation   Development nDCG@5 
* Title + Text         ------>            0.785
* Title only           ------>            0.794 
* Text only            ------>            0.732 
* Title + Title + Text  ------>            0.796

- The `Title + Title + Text` representation achieved the highest development nDCG@5 score (**0.796**) and was therefore selected for further evaluation.


- The selected `Title + Title + Text` representation was then evaluated on the held-out **58-query validation set**, which had a nDCG@5 score of **0.7712**

The configuration maintained strong performance on the validation set compared with its development score (0.796 → 0.7712), showing evidence that the selected representation generalizes reasonably well to unseen queries.

> Selected Dense Retrieval Configuration

Based on the development and validation results, `intfloat/e5-base-v2` and `Title + Title + Text` were used to configure the selected  dense-retrieval setup



After selecting the configuration, the final dense-retrieval score matrix was generated using all 308 labelled training queries. This full-query score matrix was then used as the input for the subsequent hybrid-retrieval and cross-encoder-reranker


Option 3: Hybrid Retrieval(BM25 + Dense embeddings)

In [ ]:
print(all_bm25_scores.shape)

print(cos_sim_score.shape)

(308, 695)
(308, 695)


In [ ]:
# normalize both bm25 scores and dense retireval score per query with min-max normalization and keep their dimension
def min_max_normalize(data, new_min=0, new_max=1):
   data = np.array(data, dtype=float)
   min_val = np.min(data, axis=1, keepdims=True)
   max_val = np.max(data, axis=1, keepdims=True)
   return ((data - min_val) / (max_val - min_val)) * (new_max - new_min) + new_min


norm_bm25 = min_max_normalize(all_bm25_scores)
norm_dense_ret = min_max_normalize(cos_sim_score)

In [ ]:
# split data
train_q, valid_q = train_test_split(train_queries, test_size=58,random_state=123,shuffle=True)

train_qrels = qrels_train[qrels_train["query_id"].isin(train_q["query_id"])]

val_qrels = qrels_train[qrels_train["query_id"].isin(valid_q["query_id"])]

# extract their index
devq_indices = train_q.index
validq_indices = valid_q.index

In [ ]:
devq_indices

Index([ 53,  22, 229, 107,  52, 159, 226, 115, 274,  35,
       ...
        96, 225, 214,  57, 123, 106,  83,  17, 230,  98],
      dtype='int64', length=250)

In [ ]:
# Select the same query rows from both retrieval score matrices
bm25_dev = norm_bm25[devq_indices]
bm25_valid = norm_bm25[validq_indices]

dense_dev = norm_dense_ret[devq_indices]
dense_valid = norm_dense_ret[validq_indices]

print("BM25 development:", bm25_dev.shape)
print("Dense development:", dense_dev.shape)

print("BM25 validation:", bm25_valid.shape)
print("Dense validation:", dense_valid.shape)

BM25 development: (250, 695)
Dense development: (250, 695)
BM25 validation: (58, 695)
Dense validation: (58, 695)


> > Hybrid formular: Hybrid=α(BM25)+(1−α)(Dense)

In [ ]:
alphas = [0.0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]

result = {}

for alpha in alphas:
    hybrid_score =  alpha * bm25_dev + (1 - alpha) * dense_dev

    hybrid_ndcg5 = ncdg5(
    hybrid_score,
    train_q,
    train_qrels,
    documents)

    result[alpha] = hybrid_ndcg5

In [ ]:
result

{0.0: 0.7959938393141651,
 0.05: 0.7875229604310382,
 0.1: 0.7716992031449246,
 0.2: 0.7331958413772227,
 0.3: 0.6802694950287907,
 0.4: 0.635708782593077,
 0.5: 0.5854016314972252}

* BM25 contributes nothing. Dense Retrieval alone is the best configuration among the tested weights. With the best weight(alpha) being 0

In [ ]:
# validation set with alpha set to 0
hybrid_score =  0 * bm25_valid + (1 - 0) * dense_valid

hybrid_ndcg5 = ncdg5(
    hybrid_score,
    valid_q,
    val_qrels,
    documents
    )

print(f"Hybrid validation nDCG@5: {hybrid_ndcg5:.3f}")

Hybrid validation nDCG@5: 0.771


*  BM25's contribution consistently made the development/training score worse, with the best weight(alpha) being 0.
The development/training hybrid scored **0.796**, while the validationset scored **0.771**. This shows that, for the data tested, adding BM25 scores to the Dense Retrieval scores did not improve retrieval performance. Therefore, the best-performing hybrid configuration was the Dense Retrieval alone (alpha = 0)

Cross Encoder reranking

In [ ]:
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 233.86it/s]


In [ ]:
cos_sim_score.shape

(308, 695)

In [ ]:
reranked_doc_indices = []
all_reranker_scores = []
for i in range(len(cos_sim_score)):

    #each query and its top 50 document position from dense retrieval
    query = train_queries2['query'].iloc[i]
    query_top50_indices = np.argsort(cos_sim_score[i]) [::-1] [:50]

    # query-document pairs
    pairs =[(query, doc2['text'].iloc[ind]) for ind in query_top50_indices]

    # cross encode scores the 50 candidates
    reranker_scores = reranker.predict(pairs) 
    all_reranker_scores.append(reranker_scores)

    # sort reranker scores 
    top50_reranker_indices = np.argsort(reranker_scores)[::-1]

    # get candidates actual positions/indicies 
    reranker_doc_indices = query_top50_indices[top50_reranker_indices]
    reranked_doc_indices.append(reranker_doc_indices)


In [ ]:
print(type(reranked_doc_indices))
print(len(reranked_doc_indices))
print(len(reranked_doc_indices[0]))

<class 'list'>
308
50


In [ ]:
print(type(all_reranker_scores))
print(len(all_reranker_scores))
print(len(all_reranker_scores[0]))

<class 'list'>
308
50


In [ ]:
# reranker score matrix that matches no of queries and no of documents
reranker_score_matrix = np.zeros(
    (len(train_queries2), len(doc2))
)

# place each reranker score in its corresponding document position
for i in range(len(train_queries2)):
    reranker_score_matrix[i, reranked_doc_indices[i]] = all_reranker_scores[i]

print(reranker_score_matrix.shape)

(308, 695)


In [ ]:
reranker_ndcg5 = ncdg5(
    reranker_score_matrix,
    train_queries2,
    qrels_train,
    doc2
)

print(f"Cross-Encoder Reranker nDCG@5: {reranker_ndcg5:.3f}")

Cross-Encoder Reranker nDCG@5: 0.720


Dense retrieval achieved the strongest performance among the approaches evaluated and was therefore selected as the final retrieval approach and used to generate the test-set predictions.

* Test the test_queries on seleceted retrieval model: **dense_retrieval**

In [ ]:
# encode test queries
test_emb = model2.encode(list(test_queries['query']))

# Compute cosine similarity 
test_cos_sim_score = cosine_similarity(test_emb, doc_emb)

In [ ]:
test_cos_sim_score.shape

(200, 695)

In [ ]:
# top5 doc indices
top5_indices = np.argsort(test_cos_sim_score, axis=1)[:, ::-1][:, :5]

In [ ]:
#extract first query and its top5 doc
query = test_queries['query'].iloc[0]
print(f"Query: {query}")

#query's top doc
doc_indices = top5_indices[0]


for ind in doc_indices:
    print(f"Doc_id: {doc2['document_id'].iloc[ind]} "
          f"Title: {doc2['title'].iloc[ind]}")


Query: How do I cope with drought and erratic rainfall on my farm?
Doc_id: 3 Title: Adapting to drought and erratic rainfall (Highlands)
Doc_id: 4 Title: Adapting to drought and erratic rainfall (Humid forest)
Doc_id: 5 Title: Adapting to drought and erratic rainfall (Sahel)
Doc_id: 2 Title: Adapting to drought and erratic rainfall (Guinea savanna)
Doc_id: 8 Title: Adapting to flooding and excess rain (Highlands)


* Submition of result

In [ ]:
result = []

for ind, ID in enumerate(test_queries['query_id']):
    doc_position = top5_indices[ind]

    for indx in doc_position:
        result.append({
            "QueryId": ID,
            "DocumentId": int(doc2["document_id"].iloc[indx])
        })

In [ ]:
result

[{'QueryId': 1001, 'DocumentId': 3},
 {'QueryId': 1001, 'DocumentId': 4},
 {'QueryId': 1001, 'DocumentId': 5},
 {'QueryId': 1001, 'DocumentId': 2},
 {'QueryId': 1001, 'DocumentId': 8},
 {'QueryId': 1002, 'DocumentId': 4},
 {'QueryId': 1002, 'DocumentId': 3},
 {'QueryId': 1002, 'DocumentId': 5},
 {'QueryId': 1002, 'DocumentId': 2},
 {'QueryId': 1002, 'DocumentId': 10},
 {'QueryId': 1003, 'DocumentId': 1},
 {'QueryId': 1003, 'DocumentId': 6},
 {'QueryId': 1003, 'DocumentId': 3},
 {'QueryId': 1003, 'DocumentId': 4},
 {'QueryId': 1003, 'DocumentId': 5},
 {'QueryId': 1004, 'DocumentId': 1},
 {'QueryId': 1004, 'DocumentId': 16},
 {'QueryId': 1004, 'DocumentId': 6},
 {'QueryId': 1004, 'DocumentId': 3},
 {'QueryId': 1004, 'DocumentId': 5},
 {'QueryId': 1005, 'DocumentId': 13},
 {'QueryId': 1005, 'DocumentId': 12},
 {'QueryId': 1005, 'DocumentId': 14},
 {'QueryId': 1005, 'DocumentId': 15},
 {'QueryId': 1005, 'DocumentId': 11},
 {'QueryId': 1006, 'DocumentId': 13},
 {'QueryId': 1006, 'DocumentId

In [ ]:
submission = pd.DataFrame(result)

print(submission.head(10))
print(submission.shape)
print(submission.columns)

   QueryId  DocumentId
0     1001           3
1     1001           4
2     1001           5
3     1001           2
4     1001           8
5     1002           4
6     1002           3
7     1002           5
8     1002           2
9     1002          10
(1000, 2)
Index(['QueryId', 'DocumentId'], dtype='object')


In [ ]:
submission.to_csv('submission.csv', index=False)